[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/058.%20The%20ABC_XYZ%20Inventory%20Classification%20Problem/P58-Tier-1_executed.ipynb)

---


# 58. The ABC/XYZ Inventory Classification Problem

## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Purpose and scope

This notebook classifies a set of stock keeping units (SKUs) along two dimensions: annual consumption value and historical demand variability. The ABC dimension ranks SKUs by annual value and uses cumulative-value targets to set value cutoffs. The XYZ dimension classifies demand using the coefficient of variation (CV). Combining the two labels gives one of nine possible categories, such as AX or CZ.

The inputs are each SKU's annual consumption value and a sequence of historical monthly demand observations, together with the ABC cumulative-value targets and XYZ CV cutoffs. The outputs are the two class labels, their combined label, a diagnostic confidence score, summary counts, and visualizations. The example uses the data and thresholds defined in the code cells below.

This is a deterministic rule-based classification, not an optimization problem: there is no objective function to minimize or maximize, and no solver is used. The classification rules ensure that every valid SKU receives exactly one ABC label and one XYZ label. The ABC cutoffs are determined by cumulative value; they do not impose fixed SKU-count proportions. As a result, the often-cited 20/30/50 SKU split is not guaranteed.

### Modeling assumptions

- Annual consumption values are finite, nonnegative, and expressed in the same monetary unit.
- Monthly demand observations are finite, nonnegative, and measured in units over equal-length periods. At least two observations are required to calculate a sample standard deviation.
- The XYZ coefficient of variation uses the sample standard deviation. If average demand is zero, CV is defined as infinity, placing the SKU in class Z for positive finite thresholds.
- ABC cutoffs are the annual values of the SKUs where the cumulative value first reaches each target. When values tie at a cutoff, all SKUs with that value receive the same class; therefore, the cumulative percentages may pass the targets by different amounts.
- The confidence score is a rule-based diagnostic derived from distance to class boundaries. It is not a probability, a statistical confidence interval, or a measure of forecast accuracy.

### Method and learning sequence

1. Calculate demand averages, sample standard deviations, and CV values for each SKU.
2. Sort SKUs by annual value and find the value cutoffs associated with the specified cumulative-value targets.
3. Assign ABC and XYZ labels from those cutoffs and combine the labels.
4. Review the detailed results and plots, then verify that the labels follow the stated rules.

The example includes a premium laptop, an office chair, and a seasonal decoration, along with two additional SKUs to make the summaries and plots more informative. Classification outcomes depend on all SKUs in the example because the ABC value cutoffs are calculated from the entire set.

### How to interpret the results

A-class SKUs account for the highest-value portion of the ranked list, B-class SKUs fall between the two value cutoffs, and C-class SKUs fall below the second cutoff. X indicates relatively stable demand, Y indicates intermediate variability, and Z indicates higher variability. These labels support differentiated inventory review and policy discussions; the labels alone do not prescribe a specific replenishment policy.

The notebook's verification checks confirm that classifications match the implemented rules. They do not establish that the chosen thresholds are appropriate for every organization or that historical demand will predict future demand.

## Mathematical formulation

### Demand statistics and XYZ classes

For SKU $i$, let $d_{it}$ be its demand in month $t$, where $t=1,\ldots,T_i$. The average demand, sample standard deviation, and coefficient of variation are

$$
\bar{d}_i = \frac{1}{T_i}\sum_{t=1}^{T_i} d_{it},
\qquad
s_i = \sqrt{\frac{1}{T_i-1}\sum_{t=1}^{T_i}(d_{it}-\bar{d}_i)^2},
\qquad
CV_i = \begin{cases}
s_i/\bar{d}_i, & \bar{d}_i > 0,\\
\infty, & \bar{d}_i = 0.
\end{cases}
$$

The sample standard deviation describes the spread of the observed monthly demands. Dividing it by average demand makes variability relative to the SKU's demand scale. In the code, these quantities are calculated in `SKU.__post_init__` and stored as `avg_demand`, `std_demand`, and `cv`.

Given thresholds $\gamma_1 < \gamma_2$, the XYZ class is

$$
XYZ_i =
\begin{cases}
X, & CV_i \leq \gamma_1,\\
Y, & \gamma_1 < CV_i \leq \gamma_2,\\
Z, & CV_i > \gamma_2.
\end{cases}
$$

Here, $\gamma_1$ and $\gamma_2$ are the CV cutoffs set in `ClassificationThresholds`. The inclusive upper bounds make each finite CV belong to exactly one class; an infinite CV belongs to Z.

### Cumulative-value ABC cutoffs

Let $v_i$ be the annual consumption value of SKU $i$. Sort the $n$ SKUs in non-increasing order of value, writing their values as $v_{(1)} \geq v_{(2)} \geq \cdots \geq v_{(n)}$. Define the cumulative value share after rank $k$ by

$$
C_k = \frac{\sum_{j=1}^{k} v_{(j)}}{\sum_{j=1}^{n} v_{(j)}}.
$$

For cumulative-value targets $\alpha$ and $\beta$, where $0 < \alpha < \beta \leq 1$, define

$$
k_\alpha = \min\{k : C_k \geq \alpha\},
\qquad
k_\beta = \min\{k : C_k \geq \beta\},
\qquad
a = v_{(k_\alpha)},
\qquad
b = v_{(k_\beta)}.
$$

The values $a$ and $b$ are the A and B value cutoffs returned by `calculate_abc_thresholds`. The corresponding class rule is

$$
ABC_i =
\begin{cases}
A, & v_i \geq a,\\
B, & b \leq v_i < a,\\
C, & v_i < b.
\end{cases}
$$

The first threshold is the value of the SKU at which the cumulative share reaches or exceeds $\alpha$; the second is defined in the same way for $\beta$. These rules classify by value cutoffs, not by fixed SKU-count percentages. Equal values at a cutoff remain in the same class, so a target can be exceeded, and a class can be empty in a small or tied dataset. The code implements the sorting and cumulative shares in `calculate_abc_thresholds`, then applies the value rules in `classify_sku_abc`.

### Combined classification and computational work

For every SKU, the combined label is the concatenation of its ABC and XYZ labels, for example, A followed by X gives AX. The mathematical rule assigns exactly one label from each dimension, so the combined label is one of AX, AY, AZ, BX, BY, BZ, CX, CY, or CZ. The function `mathematical_abc_xyz_classification` performs these steps for the full SKU list.

For $n$ SKUs and $T$ demand observations per SKU, calculating demand statistics takes $O(nT)$ work. Sorting for ABC cutoffs takes $O(n\log n)$ work, and assigning the labels takes $O(n)$ work. Thus, the total work is $O(nT+n\log n)$, with $O(n)$ additional storage for the sorted list and results. This is a direct computation of the stated rules, not a search over alternative solutions.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A consistent plotting style helps distinguish thresholds and demand patterns.
plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported successfully for ABC/XYZ classification.")

In [ ]:
@dataclass
class SKU:
    """Represent one SKU and its historical monthly demand."""

    sku_id: str
    annual_value: float
    monthly_demands: List[float]
    avg_demand: float = field(init=False)
    std_demand: float = field(init=False)
    cv: float = field(init=False)

    def __post_init__(self):
        """Validate the observations and calculate demand statistics."""
        if not self.sku_id:
            raise ValueError("sku_id must be a non-empty string.")
        if not np.isfinite(self.annual_value) or self.annual_value < 0:
            raise ValueError("annual_value must be finite and nonnegative.")
        if len(self.monthly_demands) < 2:
            raise ValueError("At least two monthly demand observations are required.")

        demand = np.asarray(self.monthly_demands, dtype=float)
        if not np.all(np.isfinite(demand)) or np.any(demand < 0):
            raise ValueError("Monthly demands must be finite and nonnegative.")

        self.avg_demand = float(np.mean(demand))
        self.std_demand = float(np.std(demand, ddof=1))
        self.cv = (
            self.std_demand / self.avg_demand
            if self.avg_demand > 0
            else float("inf")
        )


@dataclass
class ClassificationThresholds:
    """Store cumulative-value targets and XYZ CV cutoffs."""

    abc_alpha: float = 0.80
    abc_beta: float = 0.95
    xyz_gamma1: float = 0.10
    xyz_gamma2: float = 0.25

    def __post_init__(self):
        """Require ordered, finite thresholds for unambiguous rules."""
        values = [
            self.abc_alpha,
            self.abc_beta,
            self.xyz_gamma1,
            self.xyz_gamma2,
        ]
        if not all(np.isfinite(value) for value in values):
            raise ValueError("All classification thresholds must be finite.")
        if not 0 < self.abc_alpha < self.abc_beta <= 1:
            raise ValueError("ABC targets must satisfy 0 < alpha < beta <= 1.")
        if not 0 < self.xyz_gamma1 < self.xyz_gamma2:
            raise ValueError("XYZ cutoffs must satisfy 0 < gamma1 < gamma2.")


@dataclass
class ClassificationResult:
    """Store the classification and diagnostic score for one SKU."""

    sku: SKU
    abc_class: str
    xyz_class: str
    combined_class: str
    confidence: float


print("Data classes defined for ABC/XYZ classification.")

## Classification functions

The next functions translate the mathematical rules into code. First, the ABC thresholds are calculated from the ranked cumulative-value shares. Then each SKU is classified independently on value and demand variability. Keeping the threshold calculation and the two classification rules in separate functions makes each step easy to inspect and verify.

In [ ]:
def calculate_abc_thresholds(
    skus: List[SKU],
    thresholds: ClassificationThresholds,
) -> Tuple[float, float]:
    """Return the value cutoffs at the requested cumulative-value targets."""
    if not skus:
        raise ValueError("At least one SKU is required to calculate ABC cutoffs.")

    sorted_skus = sorted(skus, key=lambda sku: sku.annual_value, reverse=True)
    total_value = sum(sku.annual_value for sku in sorted_skus)
    if total_value <= 0:
        raise ValueError("The total annual value must be greater than zero.")

    cumulative_shares = []
    running_total = 0.0
    for sku in sorted_skus:
        running_total += sku.annual_value
        cumulative_shares.append(running_total / total_value)

    a_threshold_idx = next(
        index
        for index, share in enumerate(cumulative_shares)
        if share >= thresholds.abc_alpha
    )
    b_threshold_idx = next(
        index
        for index, share in enumerate(cumulative_shares)
        if share >= thresholds.abc_beta
    )

    a_threshold = sorted_skus[a_threshold_idx].annual_value
    b_threshold = sorted_skus[b_threshold_idx].annual_value
    return a_threshold, b_threshold


def classify_sku_abc(sku: SKU, a_threshold: float, b_threshold: float) -> str:
    """Assign A, B, or C using the annual-value cutoffs."""
    if sku.annual_value >= a_threshold:
        return "A"
    if sku.annual_value >= b_threshold:
        return "B"
    return "C"


def classify_sku_xyz(
    sku: SKU,
    thresholds: ClassificationThresholds,
) -> str:
    """Assign X, Y, or Z using the SKU's coefficient of variation."""
    if sku.cv <= thresholds.xyz_gamma1:
        return "X"
    if sku.cv <= thresholds.xyz_gamma2:
        return "Y"
    return "Z"


print("Classification functions defined.")

## Apply the mathematical classification

The combined function calculates the shared ABC cutoffs once, then applies the ABC and XYZ rules to every SKU. It also calculates a diagnostic confidence score. The score is kept separate from the class assignment: it describes proximity to the rule boundaries but does not change the assigned category.

In [ ]:
def mathematical_abc_xyz_classification(
    skus: List[SKU],
    thresholds: ClassificationThresholds,
) -> List[ClassificationResult]:
    """Classify each SKU using the specified ABC and XYZ threshold rules."""
    if not skus:
        raise ValueError("At least one SKU is required for classification.")

    a_threshold, b_threshold = calculate_abc_thresholds(skus, thresholds)
    results = []

    for sku in skus:
        abc_class = classify_sku_abc(sku, a_threshold, b_threshold)
        xyz_class = classify_sku_xyz(sku, thresholds)
        combined_class = abc_class + xyz_class
        confidence = calculate_classification_confidence(
            sku,
            a_threshold,
            b_threshold,
            thresholds,
        )

        results.append(
            ClassificationResult(
                sku=sku,
                abc_class=abc_class,
                xyz_class=xyz_class,
                combined_class=combined_class,
                confidence=confidence,
            )
        )

    return results


def calculate_classification_confidence(
    sku: SKU,
    a_threshold: float,
    b_threshold: float,
    thresholds: ClassificationThresholds,
) -> float:
    """Calculate a bounded, rule-based score from distance to class cutoffs."""
    if np.isclose(a_threshold, b_threshold):
        # Coincident value cutoffs leave no value range for class B.
        abc_confidence = 0.5
    elif sku.annual_value >= a_threshold:
        abc_confidence = min(
            1.0,
            (sku.annual_value - a_threshold) / a_threshold + 0.5,
        )
    elif sku.annual_value >= b_threshold:
        abc_confidence = 0.5 + 0.5 * (
            (sku.annual_value - b_threshold) / (a_threshold - b_threshold)
        )
    elif b_threshold > 0:
        abc_confidence = max(
            0.0,
            0.5 * sku.annual_value / b_threshold,
        )
    else:
        abc_confidence = 0.0

    if sku.cv <= thresholds.xyz_gamma1:
        xyz_confidence = 1.0 - (sku.cv / thresholds.xyz_gamma1) * 0.3
    elif sku.cv <= thresholds.xyz_gamma2:
        xyz_confidence = 0.7 - (
            (sku.cv - thresholds.xyz_gamma1)
            / (thresholds.xyz_gamma2 - thresholds.xyz_gamma1)
        ) * 0.4
    else:
        xyz_confidence = max(
            0.3,
            0.3 - (sku.cv - thresholds.xyz_gamma2) * 0.1,
        )

    return float(np.clip((abc_confidence + xyz_confidence) / 2, 0.0, 1.0))


print("Mathematical classification functions defined.")

### Understanding the confidence score

The code assigns an ABC score based on annual value's position relative to the two value cutoffs and an XYZ score based on CV's position relative to the two variability cutoffs. It averages those two scores and bounds the result between 0 and 1. A score near 1 indicates greater distance from the relevant boundaries under these particular scoring rules; a score near 0 indicates less. Because this is a designed diagnostic scale rather than a probability model, it should not be interpreted as a statistical probability that a classification is correct.

In [ ]:
# The example keeps the stated annual values and monthly demand observations.
sku1 = SKU(
    sku_id="PREMIUM_LAPTOP",
    annual_value=450000,
    monthly_demands=[85, 82, 88, 84, 86, 85, 87, 83, 85, 86, 84, 85],
)

sku2 = SKU(
    sku_id="OFFICE_CHAIR",
    annual_value=45000,
    monthly_demands=[25, 22, 35, 28, 31, 26, 29, 24, 33, 27, 30, 26],
)

sku3 = SKU(
    sku_id="SEASONAL_DECOR",
    annual_value=5000,
    monthly_demands=[2, 1, 0, 5, 8, 15, 45, 62, 35, 12, 3, 1],
)

sku4 = SKU(
    sku_id="BUDGET_PHONE",
    annual_value=125000,
    monthly_demands=[120, 118, 125, 122, 119, 121, 123, 120, 124, 121, 118, 119],
)

sku5 = SKU(
    sku_id="LUXURY_WATCH",
    annual_value=280000,
    monthly_demands=[15, 14, 16, 15, 17, 14, 15, 16, 14, 15, 16, 15],
)

example_skus = [sku1, sku2, sku3, sku4, sku5]

thresholds = ClassificationThresholds(
    abc_alpha=0.80,
    abc_beta=0.95,
    xyz_gamma1=0.10,
    xyz_gamma2=0.25,
)

print(f"Prepared {len(example_skus)} SKUs for classification.")
print(
    "Thresholds: "
    f"ABC cumulative-value targets {thresholds.abc_alpha:.0%} and "
    f"{thresholds.abc_beta:.0%}; "
    f"XYZ CV cutoffs {thresholds.xyz_gamma1:.0%} and "
    f"{thresholds.xyz_gamma2:.0%}."
)

## Detailed classifications and summaries

The following cell reports the statistics and labels for each SKU, followed by the class counts. The percentages in the summaries describe this example's observed SKU counts; they are outcomes of the data and cutoff rules, not target proportions imposed by the model.

In [ ]:
classification_results = mathematical_abc_xyz_classification(
    example_skus,
    thresholds,
)

print("DETAILED CLASSIFICATION RESULTS")
print("=" * 60)

for result in classification_results:
    sku = result.sku
    print(f"\n{sku.sku_id}:")
    print(f"  Annual value: ${sku.annual_value:,.0f}")
    print(f"  Average demand: {sku.avg_demand:.1f} units/month")
    print(f"  Sample standard deviation: {sku.std_demand:.2f} units/month")
    print(f"  Coefficient of variation: {sku.cv:.3f} ({sku.cv:.1%})")
    print(f"  ABC class: {result.abc_class}")
    print(f"  XYZ class: {result.xyz_class}")
    print(f"  Combined class: {result.combined_class}")
    print(
        "  Rule-based diagnostic score: "
        f"{result.confidence:.3f} ({result.confidence:.1%})"
    )

abc_counts = {label: 0 for label in "ABC"}
xyz_counts = {label: 0 for label in "XYZ"}
combined_counts = {}

for result in classification_results:
    abc_counts[result.abc_class] += 1
    xyz_counts[result.xyz_class] += 1
    combined_counts[result.combined_class] = (
        combined_counts.get(result.combined_class, 0) + 1
    )

total_skus = len(classification_results)
print("\nCLASSIFICATION SUMMARY")
print("=" * 60)

print("\nABC distribution:")
for abc_class in "ABC":
    count = abc_counts[abc_class]
    print(f"  {abc_class}: {count} SKUs ({count / total_skus:.1%})")

print("\nXYZ distribution:")
for xyz_class in "XYZ":
    count = xyz_counts[xyz_class]
    print(f"  {xyz_class}: {count} SKUs ({count / total_skus:.1%})")

print("\nCombined ABC/XYZ distribution:")
for combined_class in sorted(combined_counts):
    count = combined_counts[combined_class]
    print(f"  {combined_class}: {count} SKUs ({count / total_skus:.1%})")

## Visual analysis

The four panels connect the numerical rules to the example data: the cumulative-value curve shows where the ABC targets are reached, the CV chart shows the XYZ boundaries, the matrix shows the combined class counts, and the demand plot shows the monthly observations behind each CV. The plots are descriptive; they do not introduce additional classification rules.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "ABC/XYZ Inventory Classification Analysis",
    fontsize=16,
    fontweight="bold",
)

# Rank values in the same order used to calculate the ABC cutoffs.
sorted_skus = sorted(
    example_skus,
    key=lambda sku: sku.annual_value,
    reverse=True,
)
total_value = sum(sku.annual_value for sku in sorted_skus)
cumulative_shares = np.cumsum(
    [sku.annual_value for sku in sorted_skus]
) / total_value
sku_ranks = np.arange(1, len(sorted_skus) + 1)

ax1 = axes[0, 0]
ax1.plot(
    sku_ranks,
    cumulative_shares,
    color="blue",
    linewidth=2,
    marker="o",
    markersize=6,
    label="Cumulative value share",
)
ax1.axhline(
    y=thresholds.abc_alpha,
    color="red",
    linestyle="--",
    label=f"A target ({thresholds.abc_alpha:.0%})",
)
ax1.axhline(
    y=thresholds.abc_beta,
    color="darkorange",
    linestyle="--",
    label=f"B target ({thresholds.abc_beta:.0%})",
)
ax1.set_xlabel("SKU rank (descending annual value)")
ax1.set_ylabel("Cumulative annual value share")
ax1.set_title("ABC Cumulative-Value Curve")
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2 = axes[0, 1]
sku_names = [sku.sku_id.replace("_", "\n") for sku in example_skus]
cv_values = [sku.cv for sku in example_skus]
bars = ax2.bar(sku_names, cv_values, color="steelblue", alpha=0.8)
ax2.axhline(
    y=thresholds.xyz_gamma1,
    color="green",
    linestyle="--",
    label=f"X cutoff ({thresholds.xyz_gamma1:.0%})",
)
ax2.axhline(
    y=thresholds.xyz_gamma2,
    color="red",
    linestyle="--",
    label=f"Y/Z cutoff ({thresholds.xyz_gamma2:.0%})",
)
ax2.set_ylabel("Coefficient of variation")
ax2.set_title("XYZ Demand Variability")
ax2.grid(True, axis="y", alpha=0.3)
ax2.legend()

for bar, cv in zip(bars, cv_values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{cv:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax3 = axes[1, 0]
matrix_data = np.zeros((3, 3), dtype=int)
abc_indices = {"A": 0, "B": 1, "C": 2}
xyz_indices = {"X": 0, "Y": 1, "Z": 2}
category_labels = [
    ["AX", "AY", "AZ"],
    ["BX", "BY", "BZ"],
    ["CX", "CY", "CZ"],
]

for result in classification_results:
    matrix_data[
        abc_indices[result.abc_class],
        xyz_indices[result.xyz_class],
    ] += 1

image = ax3.imshow(matrix_data, cmap="Blues", alpha=0.8)
fig.colorbar(image, ax=ax3, label="Number of SKUs")
ax3.set_xticks([0, 1, 2])
ax3.set_yticks([0, 1, 2])
ax3.set_xticklabels(["X (stable)", "Y (moderate)", "Z (variable)"])
ax3.set_yticklabels(["A (high value)", "B (medium value)", "C (low value)"])
ax3.set_xlabel("XYZ class")
ax3.set_ylabel("ABC class")
ax3.set_title("Combined ABC/XYZ Classification Matrix")

for row in range(3):
    for column in range(3):
        ax3.text(
            column,
            row,
            f"{matrix_data[row, column]}\n{category_labels[row][column]}",
            ha="center",
            va="center",
            color="black",
            fontweight="bold",
        )

ax4 = axes[1, 1]
months = np.arange(1, len(example_skus[0].monthly_demands) + 1)
for sku in example_skus:
    ax4.plot(
        months,
        sku.monthly_demands,
        marker="o",
        linewidth=2,
        label=f"{sku.sku_id} ({sku.cv:.1%} CV)",
        alpha=0.85,
    )

ax4.set_xlabel("Month")
ax4.set_ylabel("Demand (units/month)")
ax4.set_title("Historical Monthly Demand Patterns")
ax4.set_xticks(months)
ax4.grid(True, alpha=0.3)
ax4.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()
print("Classification visualizations created.")

## Rule and consistency checks

These checks test the implementation against the mathematical definitions. They verify the labels and the relationship between each SKU's two labels and combined label. They also check that class counts sum to the number of SKUs. No fixed ABC count split is tested, because the cumulative-value method does not impose one.

In [ ]:
print("CLASSIFICATION RULE VERIFICATION")
print("=" * 50)

a_threshold, b_threshold = calculate_abc_thresholds(example_skus, thresholds)
valid_abc_classes = {"A", "B", "C"}
valid_xyz_classes = {"X", "Y", "Z"}

for result in classification_results:
    assert result.abc_class in valid_abc_classes
    assert result.xyz_class in valid_xyz_classes
    assert result.combined_class == result.abc_class + result.xyz_class
    assert result.abc_class == classify_sku_abc(
        result.sku,
        a_threshold,
        b_threshold,
    )
    assert result.xyz_class == classify_sku_xyz(result.sku, thresholds)
    assert 0.0 <= result.confidence <= 1.0

assert len(classification_results) == len(example_skus)
assert sum(abc_counts.values()) == len(example_skus)
assert sum(xyz_counts.values()) == len(example_skus)
assert sum(combined_counts.values()) == len(example_skus)

print("Each SKU has one valid ABC class, one valid XYZ class, and one combined class.")
print("All class labels match the stated threshold rules.")
print("Class counts cover every SKU exactly once.")
print("\nABC value cutoffs:")
print(f"  A cutoff: ${a_threshold:,.0f}")
print(f"  B cutoff: ${b_threshold:,.0f}")

average_confidence = float(
    np.mean([result.confidence for result in classification_results])
)
minimum_confidence = min(
    result.confidence for result in classification_results
)
high_score_count = sum(
    result.confidence >= 0.8 for result in classification_results
)

print("\nRule-based diagnostic scores:")
print(f"  Mean score: {average_confidence:.3f}")
print(f"  Minimum score: {minimum_confidence:.3f}")
print(
    "  Scores at or above 0.8: "
    f"{high_score_count}/{len(classification_results)}"
)
print("\nClassification rule verification complete.")

## Summary and interpretation

This notebook applies a transparent, deterministic ABC/XYZ classification procedure:

- Demand summaries use the sample standard deviation and coefficient of variation.
- ABC classes use value cutoffs found from cumulative annual consumption value.
- XYZ classes use the specified CV boundaries.
- The two labels combine into one of nine possible categories.
- Verification checks test that every classification follows the implemented rules.

The example results are displayed by the detailed-results and summary cells, rather than assumed in advance. In particular, the ABC class proportions depend on the example values, cumulative targets, and any value ties; they are not constrained to a 20/30/50 split. The confidence score is a rule-based diagnostic, not a probability or a measure of predictive accuracy.

In practice, ABC/XYZ labels can help organize inventory review and support differentiated policies. Threshold selection, demand history quality, seasonality, lead times, service targets, supplier constraints, and the cost of stockouts are business considerations that this classification model does not represent. The results should therefore be interpreted as a structured classification of the supplied data, not as a complete inventory policy.